# **Tokens and Token Embeddings**

In [1]:
%%capture
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

## **Downloading and Running An LLM**


The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [4]:
prompt = "Write a professional email applying for an AI Engineer position. Mention my experience in Python, machine learning, data analytics, and teaching AI to thousands of students.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=20
)

# Print the output
print(tokenizer.decode(generation_output[0]))


Write a professional email applying for an AI Engineer position. Mention my experience in Python, machine learning, data analytics, and teaching AI to thousands of students.<|assistant|> Subject: Application for AI Engineer Position

Dear Hiring Manager,




In [5]:
print(input_ids)

tensor([[14350,   263, 10257,  4876, 15399,   363,   385,   319, 29902, 10863,
           261,  2602, 29889,   341,  2509,   590,  7271,   297,  5132, 29892,
          4933,  6509, 29892,   848, 16114,  1199, 29892,   322, 18819,   319,
         29902,   304, 17202,   310,  8041, 29889, 32001]], device='cuda:0')


In [6]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
a
professional
email
applying
for
an
A
I
Engine
er
position
.
M
ention
my
experience
in
Python
,
machine
learning
,
data
analyt
ics
,
and
teaching
A
I
to
thousands
of
students
.
<|assistant|>


In [7]:
generation_output

tensor([[14350,   263, 10257,  4876, 15399,   363,   385,   319, 29902, 10863,
           261,  2602, 29889,   341,  2509,   590,  7271,   297,  5132, 29892,
          4933,  6509, 29892,   848, 16114,  1199, 29892,   322, 18819,   319,
         29902,   304, 17202,   310,  8041, 29889, 32001,  3323,   622, 29901,
          8427,   363,   319, 29902, 10863,   261, 20627,    13,    13, 29928,
           799,   379,  8491, 15629, 29892,    13,    13]], device='cuda:0')

In [10]:
print(tokenizer.decode(263))
print(tokenizer.decode(590))
print(tokenizer.decode([4933, 848]))
print(tokenizer.decode(29892))
print(tokenizer.decode(29928))

a
my
machine data
,
D


## **Comparing Trained LLM Tokenizers**

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '255;179;186',  # Light Pink
    '255;223;186',  # Peach
    '255;255;186',  # Light Yellow
    '186;255;201',  # Mint Green
    '186;225;255',  # Sky Blue
    '218;186;255'   # Lavender
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [12]:
text = """
Large Language Models are transforming software development.
Python, SQL, and PyTorch are essential AI tools.
if score >= 90:
    print("Excellent!")
GPU + CUDA = Faster Training 🚀
OpenAI, Anthropic, and Google build frontier AI models.
"""

In [13]:
show_tokens(text, "bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[CLS] large language models are transforming software development . python , sql , and p ##yt ##or ##ch are essential ai tools . if score > = 90 : print ( " excellent ! " ) gp ##u + cu ##da = faster training [UNK] open ##ai , ant ##hr ##op ##ic , and google build frontier ai models . [SEP] 

In [14]:
show_tokens(text, "bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] Large Language Models are transforming software development . Python , S ##QL , and P ##y ##T ##or ##ch are essential AI tools . if score > = 90 : print ( " Excellent ! " ) GP ##U + C ##U ##DA = Fast ##er Training [UNK] Open ##A ##I , An ##throp ##ic , and Google build frontier AI models . [SEP] 

In [15]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 Large  Language  Models  are  transforming  software  development . 
 Python ,  SQL ,  and  Py Tor ch  are  essential  AI  tools . 
 if  score  >=  90 : 
        print (" Excellent !" ) 
 GPU  +  CU DA  =  Faster  Training  � � � 
 Open AI ,  Anthrop ic ,  and  Google  build  frontier  AI  models . 
 

In [16]:
show_tokens(text, "google/flan-t5-small")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Large Language Model s are  transforming software development . Python , SQL , and Py T or ch are essential AI tools .  if score > = 90 : print ( " Ex cell ent !" ) GPU +  CU DA = Fast er Training  <unk> Open AI , An thro pic , and Google build frontier AI models . </s> 

In [17]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



 Large  Language  Models  are  transform ing  software  development . 
 Python ,  SQL ,  and  PyTorch  are  essential  AI  tools . 
 if  score  >=   9 0 : 
     print (" Ex cellent !") 
 GPU  +  CUDA  =  F aster  Training  � � 
 Open AI ,  An th ropic ,  and  Google  build  front ier  AI  models . 
 

In [18]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 Lar ge Language Mod els are transform ing software development . 
 Python , SQL , and Py T orch are essential A I tools . 
 if score >=  9 0 : 
    print (" Ex cell ent ! ") 
 G PU + C U DA = F aster Training  � � � � 
 Open AI , Anth rop ic , and Google build front ier A I models . 
 

## **Contextualized Word Embeddings From a Language Model (Like BERT)**

In [19]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hey Ram!! How is your day Today!!', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]


config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

model.safetensors: reconstructing file:   0%|          |  0.00B /  241MB            

model.safetensors: downloading bytes:           |  0.00B            

In [20]:
output.shape

torch.Size([1, 11, 384])

In [21]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hey
 Ram
!!
 How
 is
 your
 day
 Today
!!
[SEP]


In [22]:
output

tensor([[[-3.4824e+00, -2.5497e-02, -1.2744e-01,  ..., -1.4893e-01,
          -3.0957e-01, -1.6766e-03],
         [-5.0977e-01, -8.2947e-02,  6.7383e-01,  ..., -1.1855e+00,
          -1.3416e-01, -4.5288e-01],
         [-1.9299e-01,  1.7212e-01,  3.5449e-01,  ..., -7.8271e-01,
          -1.0732e+00, -4.0381e-01],
         ...,
         [-2.9321e-01,  3.7085e-01,  2.3901e-01,  ..., -8.3447e-01,
          -5.5371e-01, -4.7241e-01],
         [-1.5020e+00,  3.7769e-01, -1.3733e-01,  ..., -2.3572e-01,
           7.2571e-02, -5.3809e-01],
         [-3.3398e+00, -7.0114e-03, -1.2006e-01,  ..., -2.1472e-01,
          -3.9233e-01, -1.8848e-01]]], dtype=torch.float16,
       grad_fn=<NativeLayerNormBackward0>)

## **Text Embeddings (For Sentences and Whole Documents)**

In [23]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Messi is the best player in football history.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
vector.shape

(768,)

## **Word Embeddings Beyond LLMs**

In [26]:
%%capture
!pip install gensim

In [27]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [28]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

## **Recommending songs by embeddings**

In [29]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [30]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [31]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [32]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('3167', 0.9976017475128174),
 ('2849', 0.9971612691879272),
 ('3116', 0.9967880249023438),
 ('3094', 0.9966095685958862),
 ('2976', 0.9958146810531616),
 ('2104', 0.9957915544509888),
 ('5634', 0.9957317113876343),
 ('2704', 0.9956432580947876),
 ('11517', 0.9955899119377136),
 ('1922', 0.9952704310417175)]

In [33]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [34]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
3167,Unchained,Van Halen
2849,Run To The Hills,Iron Maiden
3116,Communication Breakdown,Led Zeppelin
3094,Breaking The Law,Judas Priest
2976,I Don't Know,Ozzy Osbourne


In [35]:
print_recommendations(2172)

,title,artist
id,,
3167,Unchained,Van Halen
2849,Run To The Hills,Iron Maiden
3116,Communication Breakdown,Led Zeppelin
3094,Breaking The Law,Judas Priest
2976,I Don't Know,Ozzy Osbourne


In [36]:
print_recommendations(842)

,title,artist
id,,
413,If I Ruled The World (Imagine That) (w\/ Laury...,Nas
1560,In Da Club,50 Cent
5890,Low (w\/ T-Pain),Flo-Rida
12335,King Of The Dancehall,Beenie Man
5668,How We Do (w\/ 50 Cent),The Game
